使用pyautogui來完成小恐龍的遊戲機器人

小恐龍是在 chrome 中使用  chrome://dino啟動。

小恐龍程式碼

In [ ]:

import pyautogui as pyg

# ------------------------------
# 1. 遊戲啟動
# ------------------------------

# 等待 2 秒，讓使用者切換到 Chrome 小恐龍遊戲
pyg.sleep(2)

# 按下空白鍵，啟動小恐龍遊戲
pyg.press('space')


# ------------------------------
# 2. 無窮迴圈：持續監控遊戲畫面
# ------------------------------

while True:

    # 擷取目前的螢幕畫面
    im = pyg.screenshot()

    # 預設沒有偵測到障礙物
    obs = False

    # ------------------------------
    # 3. 讀取指定位置的像素顏色
    # ------------------------------

    # 檢查 X 座標 167~181，Y 座標固定為 410
    for i in range(15):

        # 取得目前像素的 RGB 顏色
        c = im.getpixel((167 + i, 410))

        # ------------------------------
        # 4. 判斷是否出現仙人掌
        # ------------------------------

        # 如果像素顏色符合指定的 RGB 值
        if c == (83, 83, 83):

            # 表示偵測到障礙物
            obs = True

            # 找到障礙物後，不必繼續檢查
            break

    # ------------------------------
    # 5. 偵測到障礙物，自動跳躍
    # ------------------------------

    if obs == True:

        # 按下空白鍵，讓小恐龍跳起來
        pyg.press('space')


寶石方塊程式碼

In [ ]:

import pyautogui as pyg
import time

# ==============================
# 1. 基本設定
# ==============================

pyg.sleep(2)

basex = 408
basey = 490
GAP = 50

SIZE = 8

# 記錄程式開始時間
start_time = time.time()

# 遊戲執行時間：60秒
TIME_LIMIT = 60


# ==============================
# 2. 取得寶石位置
# ==============================

def gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心點的座標。"""
    return (basex + c * GAP, basey + r * GAP)


# ==============================
# 3. 取得寶石顏色
# ==============================

def get_gem_color(r, c, im):
    """取得截圖中第 r 列、第 c 欄寶石中心點的顏色。"""
    x, y = gem_position(r, c)
    return im.getpixel((x, y))


# ==============================
# 4. 判斷交換後是否可以消除
# ==============================

def _can_move_in_direction(r, c, im, dr, dc):

    nr = r + dr
    nc = c + dc

    # 確認交換位置沒有超出棋盤
    if not (0 <= r < SIZE and 0 <= c < SIZE
            and 0 <= nr < SIZE and 0 <= nc < SIZE):
        return False

    # 取得兩顆寶石的顏色
    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    # 模擬交換後的顏色
    def color_after_swap(row, col):

        if (row, col) == (r, c):
            return neighbor_color

        if (row, col) == (nr, nc):
            return current_color

        return get_gem_color(row, col, im)

    # 檢查交換後的兩個位置
    for row, col in ((r, c), (nr, nc)):

        color = color_after_swap(row, col)

        # 檢查水平及垂直方向
        for line_dr, line_dc in ((0, 1), (1, 0)):

            count = 1

            # 往兩側檢查相同顏色的寶石
            for direction in (-1, 1):

                check_r = row + direction * line_dr
                check_c = col + direction * line_dc

                while (0 <= check_r < SIZE
                       and 0 <= check_c < SIZE
                       and color_after_swap(check_r, check_c) == color):

                    count += 1

                    check_r += direction * line_dr
                    check_c += direction * line_dc

            # 三顆以上即可消除
            if count >= 3:
                return True

    return False


# ==============================
# 5. 四個方向的判斷函式
# ==============================

def can_move_left(r, c, im):
    """判斷是否可以往左移動。"""
    return _can_move_in_direction(r, c, im, 0, -1)


def can_move_right(r, c, im):
    """判斷是否可以往右移動。"""
    return _can_move_in_direction(r, c, im, 0, 1)


def can_move_up(r, c, im):
    """判斷是否可以往上移動。"""
    return _can_move_in_direction(r, c, im, -1, 0)


def can_move_down(r, c, im):
    """判斷是否可以往下移動。"""
    return _can_move_in_direction(r, c, im, 1, 0)


# ==============================
# 6. 自動交換寶石
# ==============================

def swap_gem(r, c, nr, nc):

    # 取得兩顆寶石的位置
    x1, y1 = gem_position(r, c)
    x2, y2 = gem_position(nr, nc)

    # 使用滑鼠拖曳交換寶石
    pyg.moveTo(x1, y1)
    pyg.dragTo(x2, y2, duration=0.15)

    print("交換寶石：", (r, c), "->", (nr, nc))


# ==============================
# 7. 尋找可以消除的寶石
# ==============================

def find_move(im):

    for r in range(SIZE):
        for c in range(SIZE):

            # 往右移動
            if can_move_right(r, c, im):
                return (r, c, r, c + 1)

            # 往下移動
            if can_move_down(r, c, im):
                return (r, c, r + 1, c)

    return None


# ==============================
# 8. 主程式：無窮迴圈
# ==============================

while True:

    # 判斷是否超過60秒
    if time.time() - start_time >= TIME_LIMIT:
        print("時間超過60秒，遊戲停止！")
        break

    # 擷取目前螢幕畫面
    im = pyg.screenshot()

    # 尋找可以消除的移動
    move = find_move(im)

    if move is not None:

        r, c, nr, nc = move

        # 自動交換寶石
        swap_gem(r, c, nr, nc)

        # 等待遊戲消除及補充寶石
        time.sleep(0.8)

    else:

        # 如果沒有找到可以移動的寶石
        print("目前沒有可以消除的移動")
        time.sleep(0.3)


# ==============================
# 9. 程式結束
# ==============================

print("程式執行完畢！")
print("總執行時間：", round(time.time() - start_time, 2), "秒")
